In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "jupyter", "--version"])

In [1]:
import os
import pandas as pd
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
from datetime import datetime
from docxtpl import DocxTemplate
from docx2pdf import convert
from pypdf import PdfMerger
import smtplib
from email.message import EmailMessage
from dataclasses import dataclass
import datetime as dt
import numpy as np

gis = GIS("https://www.arcgis.com", "ForestryLAC", "Dspace2026#!")

#----Data draw down----
InspLayer = gis.content.get("d2a3c681d1104be9981e0eabf40ebe4e")

data = InspLayer.get_data()

target_title = "DSPACE_INSPECTIONS_AUX_LAC_EDIT"

layer_info = next(
    lyr for lyr in data["operationalLayers"]
    if lyr["title"] == target_title
)

target = next(
    sub for sub in layer_info["layers"]
    if sub["title"] == "Primary Structure"
)


fl = FeatureLayer(target["url"])

df = fl.query(where="1=1", out_fields="*").sdf

#----Data Management----
summary = (
    df.groupby(["BATTID", "STANUM"])
      .agg(
          total_count=("PROPERTYSTATUS", "size"),
          non_null_count=("PROPERTYSTATUS", lambda x: x.notna().sum())
      )
      .reset_index()
)

summary["pct"] = summary["non_null_count"] / summary["total_count"]

output_rows = []

for battid, group in summary.groupby("BATTID"):
    group = group.sort_values("STANUM")
    
    stanums = group["STANUM"].tolist()
    
    # Header row (BATTID + STANUMs)
    output_rows.append([battid] + stanums)
    
    # Row: # of Inspected (non-null count)
    output_rows.append(
        ["# of Inspected"] + group["non_null_count"].tolist()
    )
    
    # Row: # of Inspection (total count)
    output_rows.append(
        ["# of Inspection"] + group["total_count"].tolist()
    )
    
    # Row: % of completion
    output_rows.append(
        ["% of completion"] + [f"{p*100:.1f}%" for p in group["pct"]]
    )
    
    # Blank row between sections
    output_rows.append([])

report_df = pd.DataFrame(output_rows)

bn_summary = (
    df.groupby("BATTID")
      .agg(
          inspections=("PROPERTYSTATUS", "size"),
          inspected=("PROPERTYSTATUS", lambda x: x.notna().sum())
      )
      .reset_index()
)

bn_summary["pct_completion"] = (
    bn_summary["inspected"] / bn_summary["inspections"]
)

bn_summary["pct_completion"] = (
    bn_summary["pct_completion"].apply(
        lambda x: f"{x*100:.1f}%"
    )
)
bn_summary.columns = [
    "BN",
    "# of inspection",
    "# inspected",
    "% of completion"
]

#----Excel Outputs----
output_dir = Path(r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Liaison\Weekly Reports")

output_file = output_dir / "BN_Inspection_Report.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    bn_summary.to_excel(
        writer,
        sheet_name="BN Summary",
        index=False
    )

    report_df.to_excel(
        writer,
        sheet_name="Station Summary",
        index=False,
        header=False
    )

#----PDF creation for plots + final document----
output_dir = Path(r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Liaison\Weekly Reports")
output_dir.mkdir(parents=True, exist_ok=True)

##Batallion Graph
bn_plot = bn_summary.copy()

bn_plot["BN"] = bn_plot["BN"].astype(str)

bn_plot["remaining"] = (
    bn_plot["# of inspection"] - bn_plot["# inspected"]
)

bn_plot["pct_numeric"] = (
    bn_plot["# inspected"] / bn_plot["# of inspection"] * 100
)

plt.figure(figsize=(10, 6))

plt.bar(
    bn_plot["BN"],
    bn_plot["# inspected"],
    label="Completed",
    color="green"
)

plt.bar(
    bn_plot["BN"],
    bn_plot["remaining"],
    bottom=bn_plot["# inspected"],
    label="Remaining",
    color="lightgray"
)

# Add percentage labels
for _, row in bn_plot.iterrows():
    plt.text(
        row["BN"],
        row["# of inspection"] + 2,
        f"{row['pct_numeric']:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

plt.xlabel("Battalion (BN)")
plt.ylabel("Number of Inspections")
plt.title("Inspections Completion by Battalion")
plt.legend()

plt.tight_layout()


pdf_path = output_dir / "BN_Inspection_Completion.pdf"
plt.savefig(pdf_path, format="pdf")

plt.close()

print(f"Saved plot to: {pdf_path}")

##Station Graph
summary["remaining"] = (
    summary["total_count"] - summary["non_null_count"]
)

# Create clean station labels
summary["STANUM_LABEL"] = (
    summary["STANUM"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

output_pdf = Path(
    r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Liaison\Weekly Reports\Station_Completion_Report.pdf"
)

battalions = sorted(summary["BATTID"].unique())

with PdfPages(output_pdf) as pdf:

    for page_start in range(0, len(battalions), 4):

        fig, axes = plt.subplots(
            2,
            2,
            figsize=(17, 11)
        )

        axes = axes.flatten()

        page_battalions = battalions[page_start:page_start + 4]

        for ax_idx, battid in enumerate(page_battalions):

            ax = axes[ax_idx]

            group = (
                summary[summary["BATTID"] == battid]
                .sort_values("STANUM")
            )

            x = np.arange(len(group))
            
            ax.bar(
                group["STANUM_LABEL"],
                group["non_null_count"],
                color="green",
                label="Completed"
            )

            ax.bar(
                group["STANUM_LABEL"],
                group["remaining"],
                bottom=group["non_null_count"],
                color="lightgray",
                label="Remaining"
            )

            ax.set_title(f"BN {battid}")
            ax.set_xlabel("Station")
            ax.set_ylabel("Inspections")

            ax.tick_params(
                axis="x"
            )

            ax.set_xticks(x)
            ax.set_xticklabels(group["STANUM_LABEL"])
            
            # Force all battalion charts to use the same width
            ax.set_xlim(-0.5, 7.5)

            # Optional % labels
            for x, (_, row) in enumerate(group.iterrows()):
                ax.text(
                    x,
                    row["total_count"] + 1,
                    f"{row['pct']*100:.1f}%",
                    ha="center",
                    va="bottom",
                    fontsize=9,
                    fontweight="bold"
                )

        # Hide unused plots on last page
        for i in range(len(page_battalions), 4):
            axes[i].axis("off")

        handles, labels = axes[0].get_legend_handles_labels()
        
        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.94),
            ncol=2
        )

        fig.suptitle(
            "Station Inspection Completion by Battalion",
            fontsize=16,
            y=0.98
        )

        plt.tight_layout(rect=[0, 0, 1, 0.90])

        pdf.savefig(fig)
        plt.close(fig)

print(f"Saved: {output_pdf}")


working_dir = Path(
    r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Liaison\Weekly Reports"
)

os.chdir(working_dir)

template_docx = working_dir / "Cover_Page.docx"

temp_docx = working_dir / "Cover_Page_Temp.docx"
temp_pdf = working_dir / "Cover_Page_Temp.pdf"

bn_pdf = working_dir / "BN_Inspection_Completion.pdf"
station_pdf = working_dir / "Station_Completion_Report.pdf"

report_output_dir = working_dir / "Final Reports"
report_output_dir.mkdir(exist_ok=True)

final_pdf = report_output_dir / datetime.now().strftime(
    "Weekly_Report_%m-%d-%Y.pdf"
)

doc = DocxTemplate(str(template_docx))

context = {
    "report_datetime": datetime.now().strftime(
        "%B %d, %Y %I:%M %p"
    )
}

doc.render(context)
doc.save(str(temp_docx))


convert(str(temp_docx), str(temp_pdf))

merger = PdfMerger()

for pdf in [
    temp_pdf,
    bn_pdf,
    station_pdf
]:
    merger.append(str(pdf))

merger.write(str(final_pdf))
merger.close()

print(f"Created final report: {final_pdf}")

temp_docx.unlink(missing_ok=True)
temp_pdf.unlink(missing_ok=True)


#----Email Automation----
import os
import smtplib
from email.message import EmailMessage
from dataclasses import dataclass
import datetime as dt


# =========================================================
# CONFIG
# =========================================================

@dataclass
class Config:
    FROM_EMAIL = "Fire-DefensibleSpace@fire.lacounty.gov"

    REPORT_RECIPIENTS = [
        "christian.henry@fire.lacounty.gov",
        "Kathleen.DelosReyes@fire.lacounty.gov",
        "Trevor.Moore@fire.lacounty.gov"
    ]

    SMTP_SERVER = "mail.co.la.ca.us"
    SMTP_PORT = 25
    TIMEOUT_SECONDS = 30


def build_config():
    return Config()


# =========================================================
# EMAIL FUNCTION
# =========================================================

def send_weekly_report(cfg, pdf_path):
    """
    Sends the weekly report PDF as an attachment.
    """

    if not os.path.exists(pdf_path):
        raise FileNotFoundError(
            f"PDF not found: {pdf_path}"
        )

    msg = EmailMessage()

    msg["From"] = cfg.FROM_EMAIL
    msg["To"] = ", ".join(cfg.REPORT_RECIPIENTS)
    msg["Subject"] = (
        f"Topanga Weekly Inspection Report "
        f"- {dt.datetime.now():%Y-%m-%d}"
    )

    msg.set_content(
        f"""
Hello,

Attached is the latest Weekly Inspection Report.

Generated:
{dt.datetime.now():%B %d, %Y %I:%M %p}

Regards,
Los Angeles County Fire Department
Defensible Space Unit
"""
    )

    with open(pdf_path, "rb") as f:
        pdf_data = f.read()

    msg.add_attachment(
        pdf_data,
        maintype="application",
        subtype="pdf",
        filename=os.path.basename(pdf_path)
    )

    with smtplib.SMTP(
        cfg.SMTP_SERVER,
        cfg.SMTP_PORT,
        timeout=cfg.TIMEOUT_SECONDS
    ) as server:

        server.ehlo()
        server.send_message(msg)

    print("Email sent successfully.")
    print("Recipients:")
    for recipient in cfg.REPORT_RECIPIENTS:
        print(f"  {recipient}")


# =========================================================
# MAIN
# =========================================================

def main():

    cfg = build_config()

    report_folder = Path(
        r"C:\Users\e705941\OneDrive - County of Los Angeles"
        r"\Defensible Space Unit - 2026 Defensible Space Cycle"
        r"\Liaison\Weekly Reports"
    )
    
    reports = list(report_folder.glob("Weekly_Report*.pdf"))
    
    if not reports:
        raise FileNotFoundError(
            f"No Weekly_Report PDFs found in {report_folder}"
        )
    
    report_pdf = max(
        reports,
        key=lambda p: p.stat().st_mtime
    )
    
    print(f"Using report: {report_pdf.name}")
    
    send_weekly_report(cfg, str(report_pdf))


if __name__ == "__main__":
    main()

Saved plot to: C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Liaison\Weekly Reports\BN_Inspection_Completion.pdf
Saved: C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Liaison\Weekly Reports\Station_Completion_Report.pdf


  0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\e705941\AppData\Local\Temp\ipykernel_11700\3689260096.py:336: DeprecationWarning: PdfMerger is deprecated and will be removed in pypdf 5.0.0. Use PdfWriter instead.
  merger = PdfMerger()


Created final report: C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Liaison\Weekly Reports\Final Reports\Weekly_Report_06-16-2026.pdf
Using report: Weekly_Report_06-12-2026.pdf
Email sent successfully.
Recipients:
  christian.henry@fire.lacounty.gov
  Kathleen.DelosReyes@fire.lacounty.gov


In [ ]:
import os
os.system("taskkill /f /im winword.exe")